<span style='color:#0066cc'> <span style='font-family:serif'> <font size="15"> **Access Ocean Chlorophyll A Data From PACE**

The Chlorophyll-a data suite provides global, gridded composites of surface chlorophyll-a concentration (a proxy for phytoplankton biomass). CHL supports time-series and climatologies, anomaly detection, ecosystem and fisheries applications, and teaching/training, and pairs well with AVW, PAR, Kd(490), SST, and winds for environmental context. Note: retrievals in optically complex coastal/inland waters may carry higher uncertainty—refer to the mission/algorithm documentation (e.g., OCx/OCI) for details.  **Source:** [NASA Earthdata](https://www.earthdata.nasa.gov/data/catalog/ob-cloud-pace-oci-l3m-chl-3.1).

 <span style='color:#ff6666'><font size="5"> **Requirements**
1. <font size="3"> Valid Earthdata Login (EDL)

 <span style='color:#ff6666'><font size="5"> **Objectives**

### Subset a remote file

- <font size="3">**a)** By Variables
- <font size="3">**b)** By Spatial selection

### Subset multiple remote files

- <font size="3">Stream subset of data

<span style='color:#ff6666'><font size="5"> **References**

<font size="3"> **NASA Ocean Biology Processing Group. (2025)**. PACE OCI Level-3 Global Mapped Chlorophyll (CHL) Data, version 3.1 [Data set]. NASA Ocean Biology Distributed Active Archive Center. https://doi.org/10.5067/PACE/OCI/L3M/CHL/3.1


In [ ]:
import xarray as xr
import datetime as dt
import earthaccess
import matplotlib.pyplot as plt
import numpy as np
# import pydap-specific tools
from pydap.client import get_cmr_urls, open_url
from pydap.client import to_netcdf as dap_to_netcdf

# Finding OPeNDAP URLs

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Query OPeNDAP urls using NASA's CMR API**

<font size="3"> We query NASA's CMR to identify remote files covering the following time range

- <font size="3"> January 1st to March 30st (2025).




In [ ]:
PACE_ccid = "C3620140256-OB_CLOUD" # version 3.1 
time_range=[dt.datetime(2025, 1, 1), dt.datetime(2025, 3, 30)]
cmr_urls = get_cmr_urls(ccid=PACE_ccid, limit=1000, time_range=time_range) # limit by default = 50

print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

In [ ]:
cmr_urls[:5]

## Identify only those granules associated with 4km resolution

The CMR returns URLs that belong to different resolutions. We need to further filter these to that ALL urls point to a homogeneous resource.


In [ ]:
chlor_a_urls = [url for url in cmr_urls if "DAY.CHL.V3_1.chlor_a.4km" in url]
len(chlor_a_urls)

In [ ]:
chlor_a_urls[:5]

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **EDL Authentication via earthaccess and OPeNDAP**

<font size="3.5"> You can authenticate via earthaccess as demonstrated below. You must have a valid EDL account. There are two strategies for authenticating with `earthaccess`:

1. `strategy="interactive"`. This will promt your edl username-password.
2. `strategy="netrc"`. Use this if the notebook is running on an environment where a `.netrc` with your credentials is recoverable. T

<font size="3.5"> Below the default will be `netrc`, assuming the user has executed the notebook **Authenticate.ipynb**. If not, you can change the strategy to `"interactive"`.



In [ ]:
from earthaccess.exceptions import LoginStrategyUnavailable
try:
    auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials
except LoginStrategyUnavailable:
    auth = earthaccess.login(strategy="interactive", persist=True)

# pass Token Authorization to a new Session.
my_session = session=auth.get_session()


<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Accessing Metadata-ONLY with PyDAP**

<font size="3"> We can access <span style='color:#ff6666'>**OPeNDAP**</span>-produced metadata to identify the variables of interest. In particular those associated with latitude and longitude values

<font size="3"> Below need to request the <span style='color:#ff6666'>**DAP4**</span> metadata from the remote server. To specify the protocol, we have 2 options:

**1.** <font size="3"> Replace "https" with "dap4" in the URL. This works when using `Xarray`:
```python
open_url(url.replace("https","dap4"), ...)
```
**2**. <font size="3"> Specify the protocol directly (**does not work with Xarray**)
```python
open_url(url, protocol='dap4', ...)
```

<font size="3"> Below we follow option **1)** above, and **use Xarray directly** since Xarray eagerly downloads dimension data.


In [ ]:
# create an Xarray Dataset object. It eagerly downloads all dimension data, which in this case
# it facilitates our workflow since `latitude` & `longitude` are dimension data.
ds = xr.open_dataset(chlor_a_urls[0].replace("https","dap4"), session=my_session, engine='pydap')
ds


### Subset by coordinate values

<font size="3"> Consider the case of an area of interest defined by bounding box with the following values (in python):
```python
# Min/max of lon values
minLon, maxLon = -96, 10

# Min/Max of lat values
minLat, maxLat = 6, 70
```

<font size="3"> With the **DAP4** protocol, one needs to download data and identify the indexes of the array. In python this is easy and is demonstrated below. 


In [ ]:
# Min/max of lon values
minLon, maxLon = -96, 10
# Min/Max of lat values
minLat, maxLat = 6, 70

lat, lon = ds['lat'].values, ds['lon'].values

# Identify ALL lat/lon data inside the bounding box
iLon = np.where((lon>minLon)&(lon < maxLon))[0] # 1D array
iLat= np.where((lat>minLat)&(lat < maxLat))[0] # 1D array


### Define parameters to pass to pydap to stream data from remote OPeNDAP servers

<font size="3"> These are:

1. <font size="3"> List of OPeNDAP URLs.
2. <font size="3"> `output_path`: Where the data will be stored.
3. <font size="3"> `keep_variables`: A list of variables of interest. These will be downloaded. The variable names must be fully qualifying names.
4. `<font size="3"> dim_slices`: This is a dictionary where the key is the dimension name (fully qualifying) and the values is a Tuple of first and last index. These together define the `slice` that will be applied to all relevant variables.
5. <font size="3"> `session`: A (requests) session object that contains the EDL authentication information. This is the `my_session` object above. 


In [ ]:
output_path = "data/"

keep_variables = ['/lon', '/lat', "/chlor_a"]
dim_slices = {'/lat': (iLat[0], iLat[-1]), '/lon': (iLon[0], iLon[-1])}


## Stream data

In [ ]:
%%time
dap_to_netcdf(chlor_a_urls, session=my_session, keep_variables=keep_variables, dim_slices=dim_slices, output_path=output_path)

## Inspect local files

In [ ]:
nds = xr.open_mfdataset(output_path+"PACE_OCI.*.nc4", concat_dim='time', parallel=True, combine="nested")
nds